# 7. Python Web Backend — CRITICAL

A strong backend Python engineer must know at least one modern web framework deeply. For most interview and production work, that framework is FastAPI.

FastAPI matters because it combines the best parts of modern API development:
- clean route definitions
- dependency injection
- type-based validation with Pydantic
- automatic OpenAPI docs
- async-first request handling

This notebook follows the same learning style as the other notebooks:
1. explain the concept clearly
2. show the core idea in plain language
3. give runnable Python code
4. connect each topic to backend interviews and real projects

## Topics covered

- Routing
- Dependency injection
- Pydantic and request validation
- Response models
- Middleware
- Exception handlers
- Background tasks
- Async endpoints
- Authentication
- OpenAPI
- WebSockets
- Streaming responses
- Lifecycle management
- Flask comparison

---

## FastAPI mental model

FastAPI treats an API as a collection of routes, each with a clear HTTP method, request contract, and response definition.

The most important ideas are:
- routes map HTTP requests to Python functions
- validation happens before business logic
- dependencies centralize reusable logic
- async is best when the app is mainly waiting on I/O
- documentation is generated automatically from code

This is exactly the type of thinking interviewers expect from backend engineers.


## 1. Routing

Routing is the process of mapping URL paths and HTTP methods to Python functions. In a backend, routes define the public API surface.

A route answers questions like:
- which URL should the client hit?
- which HTTP method should be used?
- what function should handle the request?

Good routing keeps the API organized, predictable, and easy to maintain.


In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def read_root():
    return {"message": "hello"}

@app.get("/items/{item_id}")
def read_item(item_id: int):
    return {"item_id": item_id, "name": "Example item"}

print("Route example ready")


## 2. Dependency Injection

Dependency injection means the framework supplies reusable dependencies to a route automatically. This is useful for authentication, database sessions, configuration values, and validation logic.

Instead of repeating the same logic in every function, you define it once and ask FastAPI to inject it when needed.


In [ ]:
from fastapi import Depends, FastAPI

app = FastAPI()

def get_token(token: str | None = None):
    return token

@app.get("/secure")
def secure_route(token: str = Depends(get_token)):
    return {"token": token}

print("Dependency injection example ready")


## 3. Pydantic Request Validation

Pydantic lets you define request schemas using Python classes. FastAPI validates incoming JSON against that schema before your endpoint logic runs.

This is one of the biggest reasons FastAPI is loved in backend work: validation happens early and consistently.


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float
    is_offer: bool = False

@app.post("/items")
def create_item(item: Item):
    return {"message": "created", "item": item}

print("Pydantic validation example ready")


## 4. Response Models

Response models define the exact shape of the JSON returned by the API. They help keep the API contract predictable for clients and other services.

This makes the API cleaner and easier to document and maintain.


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float

@app.get("/items/{item_id}", response_model=Item)
def get_item(item_id: int):
    return {"name": "Laptop", "price": 999.99}

print("Response model example ready")


## 5. Middleware

Middleware runs before and after a request is processed. It is commonly used for logging, request timing, authentication checks, and CORS handling.

Middleware sits between the client request and the route logic.


In [ ]:
from fastapi import FastAPI
import time

app = FastAPI()

@app.middleware("http")
async def add_process_time_header(request, call_next):
    start = time.time()
    response = await call_next(request)
    process_time = time.time() - start
    response.headers["X-Process-Time"] = str(process_time)
    return response

print("Middleware example ready")


## 6. Exception Handling

Exception handling lets the API convert errors into clean, predictable HTTP responses. This is important because clients should receive consistent error formats instead of raw Python tracebacks.

In a real backend, this improves debugging and API usability.


In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

app = FastAPI()

class BusinessError(Exception):
    pass

@app.exception_handler(BusinessError)
async def business_error_handler(request: Request, exc: BusinessError):
    return JSONResponse(status_code=400, content={"detail": "Business rule failed"})

print("Exception handling example ready")


## 7. Background Tasks

Background tasks allow an endpoint to return quickly while another task runs afterwards. This is useful for logging, sending emails, or doing non-critical follow-up work.

It is not a replacement for a full job queue in heavy operations.


In [ ]:
from fastapi import BackgroundTasks, FastAPI

app = FastAPI()

def write_log(message: str):
    with open("logs.txt", "a") as f:
        f.write(message + "\n")

@app.post("/send")
def send_email(background_tasks: BackgroundTasks):
    background_tasks.add_task(write_log, "email sent")
    return {"message": "queued"}

print("Background task example ready")


## 8. Async Endpoints

Async endpoints are useful when a request waits on I/O, such as database calls, network requests, or file operations. They allow the app to handle other work while waiting.

This is different from CPU-heavy work, where multiprocessing is often better.


In [ ]:
from fastapi import FastAPI
import asyncio

app = FastAPI()

@app.get("/slow")
async def slow_endpoint():
    await asyncio.sleep(1)
    return {"message": "done"}

print("Async endpoint example ready")


## 9. Authentication

Authentication verifies who the user is. Authorization determines what the user is allowed to do.

In FastAPI, these are commonly handled through dependency-based auth patterns and token validation.


In [ ]:
from fastapi import Depends, FastAPI, HTTPException, status
from fastapi.security import OAuth2PasswordBearer

app = FastAPI()
auth_scheme = OAuth2PasswordBearer(tokenUrl="token")

async def get_current_user(token: str = Depends(auth_scheme)):
    if token != "secret-token":
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid token")
    return {"user": "alice"}

@app.get("/profile")
async def profile(user=Depends(get_current_user)):
    return user

print("Authentication example ready")


## 10. WebSockets

WebSockets provide a persistent two-way connection between client and server. They are used for chat apps, live dashboards, and notification systems.

Unlike a normal HTTP request, a WebSocket stays open for ongoing communication.


In [ ]:
from fastapi import FastAPI, WebSocket

app = FastAPI()

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    while True:
        message = await websocket.receive_text()
        await websocket.send_text(f"Echo: {message}")

print("WebSocket example ready")


## 11. Streaming Responses

Streaming responses send data gradually instead of all at once. This is useful for large files, logs, AI token output, and real-time data feeds.

It helps clients start consuming results before the entire response is complete.


In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

app = FastAPI()

async def generate_numbers():
    for i in range(5):
        yield f"{i}\n"

@app.get("/stream")
async def stream():
    return StreamingResponse(generate_numbers(), media_type="text/plain")

print("Streaming response example ready")


## 12. Lifecycle Management

Lifecycle hooks are used for startup and shutdown tasks, such as database connection setup or cleanup. They help manage application resources carefully.

This is especially important in production services that need clean initialization and teardown.


In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.on_event("startup")
async def startup_event():
    print("App starting...")

@app.on_event("shutdown")
async def shutdown_event():
    print("App shutting down...")

print("Lifecycle example ready")


C:\Users\kusol\AppData\Local\Temp\ipykernel_19252\456686440.py:6: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
C:\Users\kusol\AppData\Local\Temp\ipykernel_19252\456686440.py:10: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("shutdown")


## 13. OpenAPI and Flask Comparison

OpenAPI is a standard format for describing APIs. FastAPI automatically generates this schema from your routes and Pydantic models, which is why the /docs page is so useful.

Flask is simpler and lightweight, but it does not provide the same built-in validation and documentation structure by default.

FastAPI is usually better for production APIs. Flask is excellent for small, quick services.


In [ ]:
from fastapi import FastAPI

openapi_app = FastAPI(title="OpenAPI Demo")

@openapi_app.get("/health", summary="Health check")
def health_check():
    return {"status": "ok"}

schema = openapi_app.openapi()
print(schema["info"]["title"])
print(list(schema["paths"].keys()))

from flask import Flask

flask_app = Flask(__name__)

@flask_app.route("/hello", methods=["GET"])
def hello():
    return {"message": "hello from flask"}

with flask_app.test_client() as client:
    print(client.get("/hello").get_json())

print("OpenAPI and Flask comparison ready")


OpenAPI Demo
['/health']
{'message': 'hello from flask'}
